In [1]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"



# LLM Inference Workshop – Part 6
## Supervised Fine-Tuning with QLoRA (4-bit)

In this notebook we will:

1. Explain 4-bit quantization and why QLoRA works.
2. Load a base model in 4-bit with bitsandbytes.
3. Attach LoRA adapters with PEFT.
4. Fine-tune with `Trainer`.
5. Evaluate before and after fine-tuning.



## What Is 4-Bit Quantization?

**4-bit quantization** stores model weights using 4-bit integers instead of 16/32-bit floats. In QLoRA, we keep a high-precision copy of the **small trainable LoRA adapters**, while the **frozen base model** is quantized. This reduces VRAM dramatically while still allowing gradient-based updates through the adapter layers.

**Pros**
- Much lower GPU memory footprint (enables larger models on a single GPU).
- Faster model loading and potentially higher batch sizes.
- LoRA adapters remain in higher precision, preserving trainability.

**Cons**
- Quantization introduces approximation error in the frozen base weights.
- Some operations require dequantization at runtime, adding overhead.
- Training stability can be slightly more sensitive to hyperparameters.



## Practical Note

If your GPU memory is limited, reduce:
- `TRAIN_SAMPLES`
- `MAX_LENGTH`
- `per_device_train_batch_size`
- `gradient_accumulation_steps`


In [ ]:
import os
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from IPython.display import Markdown

from utils import (
    evaluate_model,
    compute_accuracy,
    build_gsm8k_sft_datasets,
    SFTDataCollator,
)

from vllm import LLM, SamplingParams, TokensPrompt
from vllm.lora.request import LoRARequest

In [3]:

SEED = 0

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" # HuggingFaceTB/SmolLM2-135M-Instruct
OUTPUT_DIR = "artifacts/notebook6-sft-qlora-llama-1b"

TRAIN_SAMPLES = None
EVAL_SAMPLES = None
EVAL_GENERATION_SAMPLES = 32

MAX_LENGTH = 1024
MAX_NEW_TOKENS = 512

LEARNING_RATE = 2e-4
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
GRADIENT_CHECKPOINTING = True
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
LOGGING_STEPS = 10
SAVE_STEPS = 50

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]

torch.manual_seed(SEED)


In [4]:

if torch.cuda.is_available():
    device = "cuda"
    compute_dtype = torch.bfloat16
else:
    device = "cpu"
    compute_dtype = torch.float32

print("Device:", device)
print("Compute dtype:", compute_dtype)


Device: cuda
Compute dtype: torch.bfloat16


In [5]:
if device != "cuda":
    raise RuntimeError("QLoRA with bitsandbytes requires CUDA.")



## Load the Tokenizer


In [6]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


Tokenizer loaded.
Pad token: <|eot_id|>
EOS token: <|eot_id|>



## Load the Model in 4-bit

We load the base model with 4-bit NF4 quantization using bitsandbytes.
The base weights are frozen, and LoRA adapters are trained on top.


In [7]:

# 4-bit quantization config (NF4) for QLoRA
bnb_config = BitsAndBytesConfig(
    # Store base weights in 4-bit and dequantize on the fly
    load_in_4bit=True,
    # NF4 is a strong default for LLM weights
    bnb_4bit_quant_type="nf4",
    # Compute dtype for matmuls after dequantization
    bnb_4bit_compute_dtype=compute_dtype,
    # Second-stage quantization reduces memory further
    bnb_4bit_use_double_quant=True,
    # Storage dtype for quantized weights metadata
    bnb_4bit_quant_storage=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Model loaded in 4-bit.")


[2026-04-12 18:50:20] INFO modeling.py:987: We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Model loaded in 4-bit.



## Attach LoRA Adapters (QLoRA)

With a 4-bit base model, we prepare the model for k-bit training and then
attach LoRA adapters. Only adapter weights are trainable.


In [8]:

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()

# Prepare a 4-bit model for training (enable input grads, etc.)
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 6,815,744 || all params: 1,242,630,144 || trainable%: 0.5485



## Build the Training and Evaluation Datasets

We reuse the GSM8K SFT utilities from `utils.py`.


In [9]:

train_dataset, eval_dataset = build_gsm8k_sft_datasets(
    tokenizer=tokenizer,
    train_path="gsm8k/train.jsonl",
    eval_path="gsm8k/test.jsonl",
    train_samples=TRAIN_SAMPLES,
    eval_samples=EVAL_SAMPLES,
    seed=SEED,
    max_length=MAX_LENGTH,
)

data_collator = SFTDataCollator(pad_token_id=tokenizer.pad_token_id)

print(train_dataset)
print(eval_dataset)


Dataset({
    features: ['y_true', 'prompt_ids', 'input_ids', 'labels'],
    num_rows: 7473
})
Dataset({
    features: ['y_true', 'prompt_ids', 'input_ids', 'labels'],
    num_rows: 1319
})



## Baseline Evaluation


In [10]:

base_model = model
baseline_subset, baseline_responses, baseline_metrics = evaluate_model(
    base_model,
    tokenizer,
    eval_dataset,
    max_new_tokens=MAX_NEW_TOKENS,
    sample_count=3,
    batch_size=EVAL_BATCH_SIZE,
)

print(
    f"Baseline accuracy on {baseline_metrics['total']} samples: {baseline_metrics['accuracy']:.4f}"
)


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Baseline accuracy on 3 samples: 0.0000


In [11]:

print(tokenizer.decode(baseline_subset[0]["prompt_ids"]))
print()
Markdown(baseline_responses[0])


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 12 Apr 2026

You are an expert in Math.<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve the following math word problem carefully.

Think step by step to ensure the reasoning is correct.
When you are confident in the final answer, output ONLY:

\boxed{FINAL_ANSWER}

Do not include any additional text in the box.

Problem:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?<|eot_id|><|start_header_id|>assistant<|end_header_id|>





16 * 3 - 4 * 4 - 2 * 16 = 48 - 16 - 32 = 0

FINAL_ANSWER = 0


## Configure `Trainer`

We use a higher learning rate because only LoRA adapter weights are updated.


In [12]:

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    remove_unused_columns=False,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    bf16=(device != "cpu" and compute_dtype == torch.bfloat16),
    fp16=(device != "cpu" and compute_dtype == torch.float16),
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)


/tmp/ipykernel_399680/19569034.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
[2026-04-12 18:50:25] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat -fno-strict-overflow -Wsign-compare -DNDEBUG -O2 -Wall -fPIC -O2 -isystem /nfs/home/seymol/conda_envs/v3.13/include -fPIC -O2 -isystem /nfs/home/seymol/conda_envs/v3.13/include -fPIC -c /tmp/tmp10ml903h/test.c -o /tmp/tmp10ml903h/test.o
[2026-04-12 18:50:25] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat /tmp/tmp10ml903h/test.o -laio -o /tmp/tmp10ml903h/a.out
/nfs/home/seymol/conda_envs/v3.13/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
[2026-04-12 18:50:25] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat -fno-strict-overflow -Wsign-compare -DNDEBUG -O2 -Wall

[2026-04-12 18:50:26] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat /tmp/tmpd6e0fag1/test.o -laio -o /tmp/tmpd6e0fag1/a.out
/nfs/home/seymol/conda_envs/v3.13/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status



## Train


In [13]:

checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [
        os.path.join(OUTPUT_DIR, d)
        for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        checkpoint = max(checkpoints, key=os.path.getmtime)

print("Resuming from checkpoint:", checkpoint)
train_result = trainer.train(resume_from_checkpoint=checkpoint)
train_result


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Resuming from checkpoint: None


Step,Training Loss
10,0.884700
20,0.700000
30,0.639500
40,0.566300
50,0.560600
60,0.551000
70,0.535400
80,0.529900
90,0.515400
100,0.516400


TrainOutput(global_step=1170, training_loss=0.41059708962073693, metrics={'train_runtime': 1282.357, 'train_samples_per_second': 29.138, 'train_steps_per_second': 0.912, 'total_flos': 8.258324361279898e+16, 'train_loss': 0.41059708962073693, 'epoch': 5.0})


## Evaluate After Fine-Tuning


In [14]:

fine_tuned_model = trainer.model
ft_subset, ft_responses, ft_metrics = evaluate_model(
    fine_tuned_model,
    tokenizer,
    eval_dataset,
    max_new_tokens=MAX_NEW_TOKENS,
    sample_count=EVAL_GENERATION_SAMPLES,
    batch_size=EVAL_BATCH_SIZE,
)

print(
    f"Fine-tuned accuracy on {ft_metrics['total']} samples: {ft_metrics['accuracy']:.4f}"
)


Fine-tuned accuracy on 32 samples: 0.1875


In [15]:

print("Baseline accuracy:", round(baseline_metrics["accuracy"], 4))
print("Fine-tuned accuracy:", round(ft_metrics["accuracy"], 4))


Baseline accuracy: 0.0
Fine-tuned accuracy: 0.1875



## Save and Reload the QLoRA Adapter

We save only the adapter weights (plus tokenizer). To reload, attach them
onto a fresh 4-bit base model.


In [16]:

ADAPTER_DIR = os.path.join(OUTPUT_DIR, "qlora_adapter")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved QLoRA adapter to:", ADAPTER_DIR)

reloaded_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
reloaded_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
reloaded_model.eval()
reloaded_model.config.use_cache = True

print("Reloaded adapter from:", ADAPTER_DIR)


Saved QLoRA adapter to: artifacts/notebook6-sft-qlora-llama-1b/qlora_adapter


[2026-04-12 19:12:47] INFO modeling.py:987: We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Reloaded adapter from: artifacts/notebook6-sft-qlora-llama-1b/qlora_adapter


## vLLM Evaluation with LoRA (After Cleaning CUDA Memory)

Restart the kernel to free the memory.

### Important: Restart the Kernel Before vLLM

PyTorch can keep GPU memory fragments alive even after `del` and `empty_cache()`.
For vLLM evaluation, **restart the kernel** and run only
the minimal setup cells below.


In [1]:
# After restart: re-import minimal dependencies
import os

import torch
from transformers import AutoTokenizer

from utils import build_gsm8k_sft_datasets, compute_accuracy

from vllm import LLM, SamplingParams, TokensPrompt
from vllm.lora.request import LoRARequest

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [6]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" # HuggingFaceTB/SmolLM2-135M-Instruct
OUTPUT_DIR = "artifacts/notebook6-sft-qlora-llama-1b"

ADAPTER_DIR = os.path.join(OUTPUT_DIR, "qlora_adapter")

SEED = 0
MAX_LENGTH = 1024
MAX_NEW_TOKENS = 512
LORA_R = 32

train_dataset, eval_dataset = build_gsm8k_sft_datasets(
    tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME),
    train_path="gsm8k/train.jsonl",
    eval_path="gsm8k/test.jsonl",
    seed=SEED,
    max_length=MAX_LENGTH,
)

In [7]:
# Build prompts from the tokenized dataset
prompt_tokens = [TokensPrompt(prompt_token_ids=x) for x in eval_dataset["prompt_ids"]]

llm = LLM(
    model=MODEL_NAME,
    max_model_len=MAX_LENGTH + MAX_NEW_TOKENS,
    enforce_eager=True,
    gpu_memory_utilization=0.9,
    enable_lora=True,  ## Use LoRA
    max_lora_rank=LORA_R,
)

sampling_params = SamplingParams(
    max_tokens=MAX_NEW_TOKENS,
    temperature=0.0,
)

lora_request = LoRARequest("lora_adapter", 1, ADAPTER_DIR)
outputs = llm.generate(prompt_tokens, sampling_params, lora_request=lora_request)
vllm_responses = [out.outputs[0].text for out in outputs]

vllm_metrics = compute_accuracy(responses=vllm_responses, dataset=eval_dataset)
print(
    f"vLLM accuracy on {vllm_metrics['total']} samples: {vllm_metrics['accuracy']:.4f}"
)

INFO 04-12 19:36:05 [utils.py:253] non-default args: {'seed': None, 'max_model_len': 1536, 'disable_log_stats': True, 'enforce_eager': True, 'enable_lora': True, 'max_lora_rank': 32, 'model': 'meta-llama/Llama-3.2-1B-Instruct'}
INFO 04-12 19:36:06 [model.py:637] Resolved architecture: LlamaForCausalLM
INFO 04-12 19:36:06 [model.py:1750] Using max model len 1536
INFO 04-12 19:36:06 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 04-12 19:36:06 [vllm.py:601] Enforce eager set, overriding optimization level to -O0
INFO 04-12 19:36:06 [vllm.py:707] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=526946) INFO 04-12 19:36:07 [core.py:93] Initializing a V1 LLM engine (v0.12.0) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_l

(EngineCore_DP0 pid=526946) /nfs/home/seymol/conda_envs/v3.13/lib/python3.13/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:181: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=526946) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=526946)   warnings.warn(


(EngineCore_DP0 pid=526946) INFO 04-12 19:36:17 [cuda.py:411] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=526946) INFO 04-12 19:36:18 [weight_utils.py:527] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=526946) INFO 04-12 19:36:18 [default_loader.py:308] Loading weights took 0.62 seconds
(EngineCore_DP0 pid=526946) INFO 04-12 19:36:18 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore_DP0 pid=526946) INFO 04-12 19:36:21 [gpu_model_runner.py:3549] Model loading took 2.3765 GiB memory and 4.734737 seconds
(EngineCore_DP0 pid=526946) INFO 04-12 19:36:25 [gpu_worker.py:359] Available KV cache memory: 31.93 GiB
(EngineCore_DP0 pid=526946) INFO 04-12 19:36:26 [kv_cache_utils.py:1286] GPU KV cache size: 1,046,208 tokens
(EngineCore_DP0 pid=526946) INFO 04-12 19:36:26 [kv_cache_utils.py:1291] Maximum concurrency for 1,536 tokens per request: 681.12x
(EngineCore_DP0 pid=526946) INFO 04-12 19:36:28 [core.py:254] init engine (profile, create kv cache, warmup model) took 7.13 seconds
(EngineCore_DP0 pid=526946) WARNING 04-12 19:36:30 [vllm.py:608] Inductor compilation was disabled by user settings,Optimizations settings that are only active duringInductor compilation 

Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

(EngineCore_DP0 pid=526946) WARNING 04-12 19:36:32 [utils.py:250] Using default LoRA kernel configs
vLLM accuracy on 1319 samples: 0.3927


In [8]:
print(vllm_responses[0])

Janet has 16 - 3 = <<16-3=13>>13 fresh eggs left every day.
She bakes muffins for 13 - 4 = <<13-4=9>>9 days a week.
She makes 9 x $2 = $<<9*2=18>>18 a week from muffins.
Thus, Janet makes $18 x 4 = $<<18*4=72>>72 a week from the muffins.
\boxed{72}



## Summary

In this notebook we:
- explained 4-bit quantization and its tradeoffs
- loaded a base model in 4-bit with bitsandbytes
- attached LoRA adapters (QLoRA)
- fine-tuned using `Trainer`
- evaluated the fine-tuned model by generation
